# PathOGen fidelity experiments
Run this notebook in a GPU Colab runtime. L4 is the recommended cost-effective option. Execute setup and verification before the smoke test or paper experiments.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO = Path('/content/PathOGen')
if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', '--branch', 'colab-fidelity-experiments', '--single-branch', 'https://github.com/a12dongithub/PathOGen.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'colab-fidelity-experiments'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'switch', 'colab-fidelity-experiments'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
os.chdir(REPO)
subprocess.run(['nvidia-smi'], check=True)

## Mount Drive
Use Drive for the CellViT++ checkpoint and outputs that must survive a runtime reset.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## Install and prepare assets
Set either `CELLVIT_MODEL` to an existing Drive file or `CELLVIT_MODEL_URL` to a shareable Drive link. The dataset and FID58 model links are built into the setup script.

In [ ]:
CELLVIT_MODEL = '/content/drive/MyDrive/PathOGenAssets/CellViT-256-x40-AMP.pth'
CELLVIT_MODEL_URL = ''
OUTPUT_ROOT = '/content/drive/MyDrive/PathOGenResults'

command = ['python', 'experiments/colab/setup_colab.py', '--output-root', OUTPUT_ROOT]
if Path(CELLVIT_MODEL).is_file():
    command += ['--cellvit-model', CELLVIT_MODEL]
elif CELLVIT_MODEL_URL:
    command += ['--cellvit-model-url', CELLVIT_MODEL_URL]
else:
    print('CellViT++ checkpoint not configured; setup will explain where to upload it.')
subprocess.run(command, check=True)

## Verify dependencies, assets and experiment plans

In [ ]:
subprocess.run(['python', 'experiments/colab/verify_colab.py'], check=True)
subprocess.run(['python', 'experiments/colab/run_fidelity_suite.py', '--dry-run', '--num-images', '3'], check=True)

## End-to-end smoke test
This uses two denoising steps only to validate generation, CellViT++ and result writing. Do not use its image for quality evaluation.

In [ ]:
subprocess.run(['python', 'experiments/colab/run_fidelity_suite.py', '--smoke-test'], check=True)

## Paper run
Start small. With eight morphology controls, 25 cases require approximately 250 generations. Existing artifacts are resumed automatically.

In [ ]:
NUM_IMAGES = 25
STEPS = 20
subprocess.run([
    'python', 'experiments/colab/run_fidelity_suite.py',
    '--experiments', 'all',
    '--num-images', str(NUM_IMAGES),
    '--steps', str(STEPS),
    '--bootstrap', '1000',
    '--seed', '42',
], check=True)

## Baseline versus CellViT++ best-of-48 FID/KID
This script calculates baseline FID/KID, generates a balanced grid of 3 green levels by 2 spatial strengths at fixed 30-step denoising across the same 8 seeds, ranks all 48 candidates per input using only the +1/0/-1 CellViT++ 50-pixel point score, then calculates FID/KID on the selected images. Start with the dry run.

In [ ]:
subprocess.run(['python', 'experiments/05_cellvit_rerank_fid_kid.py', '--dry-run', '--num-images', '3'], check=True)

RERANK_INPUTS = 5000
subprocess.run([
    'python', 'experiments/05_cellvit_rerank_fid_kid.py',
    '--num-images', str(RERANK_INPUTS),
    '--seeds-per-config', '8',
    '--match-radius', '50',
    '--generation-batch-size', '4',
    '--cellvit-batch-size', '4',
    '--seed', '42',
], check=True)